In [100]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from colorama import Fore, Style
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [101]:
#data_dir = './EyeQ'
data_dir = './EyeQ_final_data'
base_dir = './EyeQ_final_data'


PIDs = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and d[:].isdigit()]

#Convert PIDs to int
PIDs = sorted([int(pid) for pid in PIDs])

print(f"Found {len(PIDs)} PIDs: {PIDs}")

Found 48 PIDs: [1006, 1013, 1015, 1019, 1021, 1029, 1031, 1032, 1033, 1037, 1038, 1039, 1040, 1044, 1045, 1050, 1061, 1062, 1063, 1068, 1070, 1071, 1073, 1075, 1077, 1078, 1080, 1082, 1084, 1085, 1086, 1088, 1092, 1104, 2003, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3012, 3013, 3014]


In [102]:
eyeq_data = pd.read_csv(os.path.join(base_dir, 'EyeQ Data - Master.csv'))
eyeq_data = eyeq_data[eyeq_data['Participant ID'].isin(PIDs)]

vimssq_low_PIDs = eyeq_data[eyeq_data['VIMSSQ-Intake'] < 22]['Participant ID'].unique()
vimssq_high_PIDs = eyeq_data[eyeq_data['VIMSSQ-Intake'] >= 22]['Participant ID'].unique()

In [103]:
def load_data(base_dir):
    eyeq_data = pd.read_csv(os.path.join(base_dir, 'EyeQ Data - Master.csv'))
    vimssq_data = eyeq_data[['Participant ID', 'VIMSSQ-Group-Intake_Recruit']].drop_duplicates(subset=['Participant ID'])

    print(f"Searching for data in {base_dir}...")
    all_files = []
    
    # Walk through directories
    for root, dirs, files in os.walk(base_dir):
        #Skip the "DQ" and "Dry runs"
        if 'DQ' in dirs:
            dirs.remove('DQ')
        if 'Dry runs' in dirs:
            dirs.remove('Dry runs')
            
        #Check if we are in a 'final' folder
        if os.path.basename(root) == 'final':
            #Check if the parent folder (PID) is a number
            parent_folder = os.path.basename(os.path.dirname(root))
            if parent_folder.isdigit():
                for file in files:
                    #Check if the file name contains 'responses_final'
                    if 'responses_final' in file and file.endswith(".csv"):
                        all_files.append(os.path.join(root, file))
    
    print(f"Found {len(all_files)} valid CSV files.")
    
    df_list = []
    for filename in all_files:
        try:
            temp_df = pd.read_csv(filename)

            #Print PID
            #participant_id = temp_df['Participant ID'].iloc[0] if 'Participant ID' in temp_df.columns else 'Unknown'
            #print(f"Processing PID: {participant_id} from file: {filename}")
            
            #Basic Cleaning
            temp_df['Response'] = pd.to_numeric(temp_df['Response'], errors='coerce')
            
            # Filter Missed
            if 'Missed' in temp_df.columns:
                temp_df = temp_df[temp_df['Missed'] != True]
            
            temp_df = temp_df.dropna(subset=['Response'])
            
            #Use Target Time for binning (Seconds -> Minutes)
            if 'Target Time (seconds)' in temp_df.columns:
                temp_df['Time (Minutes)'] = temp_df['Target Time (seconds)'] / 60
            else:
                # Fallback if column is missing, though it shouldn't be based on specs
                temp_df['Time (Minutes)'] = temp_df['Global Time (seconds)'] / 60
            
            temp_df = temp_df.merge(vimssq_data[['Participant ID', 'VIMSSQ-Group-Intake_Recruit']], on='Participant ID', how='left')
            temp_df = temp_df.rename(columns={'VIMSSQ-Group-Intake_Recruit': 'VIMSSQ Bin'})
            #temp_df['VIMSSQ Bin'] = pd.cut(temp_df['VIMSSQ Score'], bins = [0, 21, 165], labels = ['Low', 'High'])
            df_list.append(temp_df)

        except Exception as e:
            print(f"Error reading {filename}: {e}")

    if not df_list:
        # Create an empty dummy DF to prevent app crash on load if no data found
        return pd.DataFrame(columns=['Participant ID', 'Survey Type', 'Time (Minutes)', 'Response', 'Device Type', 'Session Type', 'VIMSSQ Score'])

    master_df = pd.concat(df_list, ignore_index=True)
    
    # Ensure categorical columns are strings for cleaner plotting
    master_df['Participant ID'] = master_df['Participant ID'].astype(str)
    return master_df

In [104]:
master_df = load_data(base_dir)

Searching for data in ./EyeQ_final_data...
Found 382 valid CSV files.


In [105]:
# Derive "VIMS Induced" categorical: Yes if participant ever hit MISC >= 4, No otherwise
misc_max = (master_df[master_df['Survey Type'] == 'misc']
            .groupby('Participant ID')['Response']
            .max()
            .reset_index()
            .rename(columns={'Response': 'Max MISC'}))
misc_max['VIMS Induced'] = misc_max['Max MISC'].apply(lambda x: 'Yes' if x >= 3 else 'No')

master_df = master_df.merge(misc_max[['Participant ID', 'VIMS Induced']], on='Participant ID', how='left')

print(f"VIMS Induced counts: {master_df.groupby('Participant ID')['VIMS Induced'].first().value_counts().to_dict()}")

VIMS Induced counts: {'No': 28, 'Yes': 20}


In [121]:
# Create a crosstab to see the relationship between VIMS Induced and VIMSSQ Bin
vims_crosstab = pd.crosstab(
    master_df.groupby('Participant ID')['VIMS Induced'].first(),
    master_df.groupby('Participant ID')['VIMSSQ Bin'].first(),
    margins=True
)

fig = go.Figure(data=go.Heatmap(
    z=vims_crosstab.iloc[:-1, :-1].values,
    x=vims_crosstab.columns[:-1],
    y=vims_crosstab.index[:-1],
    text=vims_crosstab.iloc[:-1, :-1].values,
    texttemplate='%{text}',
    textfont={"size": 16},
    colorscale='Blues',
    showscale=True
))

fig.update_layout(
    title='Cross-tabulation: VIMS Induced vs VIMSSQ Bin',
    xaxis_title='VIMSSQ Bin',
    yaxis_title='VIMS Induced',
    width=600,
    height=400,
    template='plotly_white'
)

fig.show()

In [107]:
def plot_figure_1(df, survey_type, col_var, hue_var, exclude_vomit=1):
    # Accept single string or list of survey types
    if isinstance(survey_type, str):
        survey_type = [survey_type]
    
    survey_df = df[df['Survey Type'].isin(survey_type)].copy()
    if exclude_vomit:
        survey_df = survey_df[survey_df['Response'] < 10]

    # Pre-aggregate: compute mean, SE, and n per group
    group_cols = ['Survey Type', col_var, hue_var, 'Time (Minutes)']
    grouped = survey_df.groupby(group_cols)['Response']
    agg_df = grouped.agg(['mean', 'sem', 'count']).reset_index()
    agg_df.columns = group_cols + ['Mean Response', 'SE', 'n']
    
    # Calculate upper and lower bounds for shaded area
    agg_df['Upper'] = agg_df['Mean Response'] + agg_df['SE']
    agg_df['Lower'] = agg_df['Mean Response'] - agg_df['SE']

    # Build title and facet_row setting
    title = ' / '.join(s.capitalize() for s in survey_type) + ' Scores Over Time'
    facet_row = 'Survey Type' if len(survey_type) > 1 else None

    fig = px.line(
        agg_df,
        x='Time (Minutes)',
        y='Mean Response',
        color=hue_var,
        facet_col=col_var,
        facet_row=facet_row,
        markers=True,
        title=title,
        hover_data={'SE': ':.2f', 'n': True},
    )
    
    # Share y-axes within each row (same survey type) but let different rows scale independently
    if facet_row:
        n_cols_grid = agg_df[col_var].nunique()
        for row in range(1, len(survey_type)):
            for col in range(n_cols_grid):
                axis_idx = row * n_cols_grid + col
                axis_name = f'yaxis{axis_idx + 1}'
                if col == 0:
                    fig.layout[axis_name].matches = None
                    fig.layout[axis_name].showticklabels = True
                else:
                    first_in_row = row * n_cols_grid + 1
                    fig.layout[axis_name].matches = f'y{first_in_row}'
    
    # Add shaded error bands by matching each trace's data to the correct group
    for trace in fig.data:
        hue_value = trace.name.split(',')[0].strip()
        if hue_value.isdigit():
            hue_value = int(hue_value)
        
        trace_y = list(trace.y)
        
        # Find the aggregated group whose means match this trace's y-values
        matched_data = None
        for sv in agg_df['Survey Type'].unique():
            for cv in agg_df[col_var].unique():
                subset = agg_df[
                    (agg_df[hue_var] == hue_value) &
                    (agg_df[col_var] == cv) &
                    (agg_df['Survey Type'] == sv)
                ].sort_values('Time (Minutes)')
                
                if len(subset) == len(trace_y) and np.allclose(subset['Mean Response'].values, trace_y, equal_nan=True):
                    matched_data = subset
                    break
            if matched_data is not None:
                break
        
        if matched_data is None:
            continue
        
        fig.add_trace(
            go.Scatter(
                x=matched_data['Time (Minutes)'].tolist() + matched_data['Time (Minutes)'].tolist()[::-1],
                y=matched_data['Upper'].tolist() + matched_data['Lower'].tolist()[::-1],
                fill='toself',
                fillcolor=trace.line.color,
                opacity=0.2,
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip',
                xaxis=trace.xaxis,
                yaxis=trace.yaxis
            )
        )

    height = 500 if len(survey_type) == 1 else 350 * len(survey_type)

    fig.update_layout(
        height=height, 
        width=1000, 
        template='plotly_white',
        title_x=0.5,
        title_font_size=18,
        legend=dict(
            orientation='h', 
            yanchor='top', 
            y=-0.25, 
            xanchor='center', 
            x=0.5
        ),
        legend_title_text=hue_var
    )
    fig.show()

    return agg_df, fig

In [113]:
misc_session_vims, fig1a = plot_figure_1(master_df, 'misc', 'VIMSSQ Bin', 'Session Type') #Figure 1A
eyestrain_session_vims, fig1b = plot_figure_1(master_df, 'eyestrain', 'VIMSSQ Bin', 'Session Type') #Figure 1B
misc_study_day_vims, fig1c = plot_figure_1(master_df, 'misc', 'VIMSSQ Bin', 'Study Day') #Figure 1C
eyestrain_study_day_vims, fig1d = plot_figure_1(master_df, 'eyestrain', 'VIMSSQ Bin', 'Study Day') #Figure 1D

In [116]:
misc_session_vims_induced, fig1a1 = plot_figure_1(master_df, 'misc', 'VIMS Induced', 'Session Type') #Figure 1a.1
eyestrain_session_vims_induced, fig1b1 = plot_figure_1(master_df, 'eyestrain', 'VIMS Induced', 'Session Type') #Figure 1b.1
misc_study_day_vims_induced, fig1c1 = plot_figure_1(master_df, 'misc', 'VIMS Induced', 'Study Day') #Figure 1c.1
eyestrain_study_day_vims_induced, fig1d1 = plot_figure_1(master_df, 'eyestrain', 'VIMS Induced', 'Study Day') #Figure 1d.1

In [119]:
def plot_figure_2(df, survey_type, row_var, exclude_vomit=1, hue_var=None):
    survey_df = df[df['Survey Type'] == survey_type].copy()
    if exclude_vomit:
        survey_df = survey_df[survey_df['Response'] < 10]

    survey_df['Day/Session'] = 'D' + survey_df['Study Day'].astype(str) + 'S' + survey_df['Daily Session'].astype(str)
    order = ['D1S1', 'D1S2', 'D2S1', 'D2S2', 'D3S1', 'D3S2', 'D4S1', 'D4S2']

    # Pre-aggregate: compute mean, SE, and n per group
    group_cols = [row_var, 'Day/Session']
    if hue_var:
        group_cols.append(hue_var)

    grouped = survey_df.groupby(group_cols)['Response']
    agg_df = grouped.agg(['mean', 'sem', 'count']).reset_index()
    agg_df.columns = group_cols + ['Mean Response', 'SE', 'n']

    agg_df['Upper'] = agg_df['Mean Response'] + agg_df['SE']
    agg_df['Lower'] = agg_df['Mean Response'] - agg_df['SE']

    # Enforce x-axis ordering via numeric position for shaded bands
    order_map = {v: i for i, v in enumerate(order)}
    agg_df['_x_num'] = agg_df['Day/Session'].map(order_map)

    # Put both groups on a single plot: row_var as color, hue_var as line_dash
    fig = px.line(
        agg_df,
        x='Day/Session',
        y='Mean Response',
        color=row_var,
        line_dash=hue_var,
        markers=True,
        title=f'{str.capitalize(survey_type)} Scores Over Days',
        category_orders={'Day/Session': order},
        hover_data={'SE': ':.2f', 'n': True},
    )

    # Add shaded error bands by matching each trace's data
    for trace in fig.data:
        color_value = trace.name.split(',')[0].strip()
        trace_y = list(trace.y)

        # Find matching group
        matched_data = None
        for rv in agg_df[row_var].unique():
            if hue_var:
                for hv in agg_df[hue_var].unique():
                    subset = agg_df[
                        (agg_df[row_var] == rv) &
                        (agg_df[hue_var] == hv)
                    ].sort_values('_x_num')
                    if len(subset) == len(trace_y) and np.allclose(subset['Mean Response'].values, trace_y, equal_nan=True):
                        matched_data = subset
                        break
                if matched_data is not None:
                    break
            else:
                subset = agg_df[agg_df[row_var] == rv].sort_values('_x_num')
                if len(subset) == len(trace_y) and np.allclose(subset['Mean Response'].values, trace_y, equal_nan=True):
                    matched_data = subset
                    break

        if matched_data is None:
            continue

        x_vals = matched_data['Day/Session'].tolist()
        fig.add_trace(
            go.Scatter(
                x=x_vals + x_vals[::-1],
                y=matched_data['Upper'].tolist() + matched_data['Lower'].tolist()[::-1],
                fill='toself',
                fillcolor=trace.line.color,
                opacity=0.2,
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip',
                xaxis=trace.xaxis,
                yaxis=trace.yaxis,
                marker=None
            )
        )

    fig.update_layout(height=450, width=800, template='plotly_white',
                      title_x=0.5, legend_title_text=row_var)
    fig.show()

In [120]:
plot_figure_2(master_df, 'misc', 'VIMSSQ Bin') #Figure 2A
plot_figure_2(master_df, 'eyestrain', 'VIMSSQ Bin') #Figure 2B

In [68]:
# Load SSQ data from Master CSV and reshape to long format
ssq_raw = pd.read_csv(os.path.join(base_dir, 'EyeQ Data - Master.csv'))
ssq_raw = ssq_raw[ssq_raw['Participant ID'].isin(PIDs)]

ssq_cols_pre = ['SSQ-T-Pre', 'SSQ-N-Pre', 'SSQ-O-Pre', 'SSQ-D-Pre']
ssq_cols_post = ['SSQ-T-Post', 'SSQ-N-Post', 'SSQ-O-Post', 'SSQ-D-Post']

id_cols = ['Participant ID', 'Day', 'Session']

ssq_pre = ssq_raw[id_cols + ssq_cols_pre].melt(
    id_vars=id_cols, var_name='Subscale', value_name='Score')
ssq_pre['Timing'] = 'Pre'
ssq_pre['Subscale'] = ssq_pre['Subscale'].str.replace('SSQ-', '').str.replace('-Pre', '')

ssq_post = ssq_raw[id_cols + ssq_cols_post].melt(
    id_vars=id_cols, var_name='Subscale', value_name='Score')
ssq_post['Timing'] = 'Post'
ssq_post['Subscale'] = ssq_post['Subscale'].str.replace('SSQ-', '').str.replace('-Post', '')

# Enforce pairing: only keep sessions where BOTH Pre and Post exist per subscale
pair_keys = ['Participant ID', 'Day', 'Session', 'Subscale']
pre_valid = ssq_pre.dropna(subset=['Score'])[pair_keys]
post_valid = ssq_post.dropna(subset=['Score'])[pair_keys]
valid_pairs = pre_valid.merge(post_valid, on=pair_keys)

ssq_pre = ssq_pre.merge(valid_pairs, on=pair_keys)
ssq_post = ssq_post.merge(valid_pairs, on=pair_keys)

ssq_long = pd.concat([ssq_pre, ssq_post], ignore_index=True)

subscale_labels = {'T': 'Total', 'N': 'Nausea', 'O': 'Oculomotor', 'D': 'Disorientation'}
ssq_long['Subscale'] = ssq_long['Subscale'].map(subscale_labels)

n_dropped = len(ssq_raw) * 8 - len(ssq_long)  # 8 = 4 subscales x 2 timings per row
print(f"SSQ data: {len(ssq_long)} rows across {ssq_long['Participant ID'].nunique()} participants")
print(f"Dropped {n_dropped} unpaired observations (missing Pre or Post)")

SSQ data: 3056 rows across 48 participants
Dropped 16 unpaired observations (missing Pre or Post)


In [69]:
import textwrap

def plot_ssq_paired(ssq_df, facet_var='Day', legend_wrap=20):
    sub_df = ssq_df.copy()

    # Wrap long legend labels so they fit in a fixed-width legend
    sub_df[facet_var] = sub_df[facet_var].apply(
        lambda x: '<br>'.join(textwrap.wrap(str(x), width=legend_wrap)))

    agg = sub_df.groupby(['Subscale', facet_var, 'Timing'])['Score'].agg(['mean', 'sem', 'count']).reset_index()
    agg.columns = ['Subscale', facet_var, 'Timing', 'Mean Score', 'SE', 'n']

    fig = px.line(
        agg,
        x='Timing',
        y='Mean Score',
        color=facet_var,
        facet_col='Subscale',
        error_y='SE',
        markers=True,
        title=f'SSQ Pre vs Post by {facet_var}',
        category_orders={
            'Timing': ['Pre', 'Post'],
            'Subscale': ['Total', 'Nausea', 'Oculomotor', 'Disorientation'],
        },
        hover_data={'SE': ':.2f', 'n': True},
    )

    fig.update_yaxes(matches=None, showticklabels=True)
    fig.update_traces(marker_size=10)
    fig.update_layout(
        height=400, width=1000, template='plotly_white',
        title_x=0.5,
        legend_title_text=facet_var,
        legend=dict(entrywidth=120, entrywidthmode='pixels'),
    )
    fig.show()

plot_ssq_paired(ssq_long)

In [70]:
# Derive Session Type for SSQ data by merging with responses data
# Master CSV has Session='Morning'/'Afternoon', responses have Daily Session=1/2
session_map = {'Morning': 1, 'Afternoon': 2}
ssq_with_session = ssq_raw[['Participant ID', 'Day', 'Session'] + ssq_cols_pre + ssq_cols_post].copy()
ssq_with_session['Daily Session'] = ssq_with_session['Session'].map(session_map)

# Get unique Session Type per (PID, Day, Daily Session) from responses data
session_type_lookup = master_df[['Participant ID', 'Study Day', 'Daily Session', 'Session Type']].drop_duplicates()
session_type_lookup['Participant ID'] = session_type_lookup['Participant ID'].astype(int)

ssq_with_session = ssq_with_session.merge(
    session_type_lookup,
    left_on=['Participant ID', 'Day', 'Daily Session'],
    right_on=['Participant ID', 'Study Day', 'Daily Session'],
    how='left',
)

# Drop sessions with no Session Type (missing response files) and report
missing_st = ssq_with_session[ssq_with_session['Session Type'].isna()]
if len(missing_st) > 0:
    for _, row in missing_st.iterrows():
        print(f"Dropped PID {row['Participant ID']} Day {row['Day']} {row['Session']}: "
              f"no response file to derive Session Type")
ssq_with_session = ssq_with_session.dropna(subset=['Session Type'])

id_cols_st = ['Participant ID', 'Day', 'Daily Session', 'Session Type']

ssq_pre_st = ssq_with_session[id_cols_st + ssq_cols_pre].melt(
    id_vars=id_cols_st, var_name='Subscale', value_name='Score')
ssq_pre_st['Timing'] = 'Pre'
ssq_pre_st['Subscale'] = ssq_pre_st['Subscale'].str.replace('SSQ-', '').str.replace('-Pre', '')

ssq_post_st = ssq_with_session[id_cols_st + ssq_cols_post].melt(
    id_vars=id_cols_st, var_name='Subscale', value_name='Score')
ssq_post_st['Timing'] = 'Post'
ssq_post_st['Subscale'] = ssq_post_st['Subscale'].str.replace('SSQ-', '').str.replace('-Post', '')

# Enforce pairing: only keep sessions where BOTH Pre and Post exist per subscale
pair_keys_st = ['Participant ID', 'Day', 'Daily Session', 'Subscale']
pre_valid_st = ssq_pre_st.dropna(subset=['Score'])[pair_keys_st]
post_valid_st = ssq_post_st.dropna(subset=['Score'])[pair_keys_st]
valid_pairs_st = pre_valid_st.merge(post_valid_st, on=pair_keys_st)

ssq_pre_st = ssq_pre_st.merge(valid_pairs_st, on=pair_keys_st)
ssq_post_st = ssq_post_st.merge(valid_pairs_st, on=pair_keys_st)

ssq_by_session_type = pd.concat([ssq_pre_st, ssq_post_st], ignore_index=True)
ssq_by_session_type['Subscale'] = ssq_by_session_type['Subscale'].map(subscale_labels)

print(f"\nSSQ by Session Type: {len(ssq_by_session_type)} rows")
print(f"Session Types found: {sorted(ssq_by_session_type['Session Type'].unique().tolist())}")

Dropped PID 1021 Day 1 Afternoon: no response file to derive Session Type
Dropped PID 2003 Day 1 Afternoon: no response file to derive Session Type

SSQ by Session Type: 3048 rows
Session Types found: ['Capture & Share', 'Discover & Learn', 'Setup & Device Management', 'Watching']


In [71]:
plot_ssq_paired(ssq_by_session_type, facet_var='Session Type')